# Helene: Cluster-Level Recovery Analysis

**Objective**: Using the income+geography Ward's clusters (k=20) from `helene_spatial_clustering.ipynb`,
aggregate within-region mobility per cluster, fit SARIMAX baselines, compute largest drop
and recovery time, then run regression analysis (OLS + Kruskal-Wallis) similar to Milton.

**Pipeline**:
1. Load cluster assignments and raw mobility H5 data
2. Aggregate within-cluster mobility per day
3. Fit SARIMAX per cluster → predicted baseline
4. Compute largest drop and recovery time per cluster (with plots)
5. OLS regression: drop/recovery ~ median income + distance
6. Kruskal-Wallis quartile analysis by income

In [74]:
import pandas as pd
import numpy as np
import os
import sys
import warnings
import datetime as dt
from importlib import reload

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr, pearsonr, kruskal, mannwhitneyu
from statsmodels.nonparametric.smoothers_lowess import lowess as lowess_func

warnings.filterwarnings("ignore")

# Project paths
folder_path = "./../../hurricane_oct/"
sys.path.append(folder_path)
sys.path.append(os.path.join(folder_path, "mobility_function"))
from mobility_function import analysis as ma
ma = reload(ma)

%run ./recovery_function_v2.py

OUTPUT_DIR = "../results/helene_cluster_analysis/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "figures"), exist_ok=True)
print("Setup complete.")

Setup complete.


## 1. Load Data

In [75]:
# ── Load cluster assignments ──
cluster_df = pd.read_csv("../results/helene_clustering/county_cluster_assignments.csv")
cluster_df["GEOID"] = cluster_df["GEOID"].astype(int)

# ── Load county index mapping ──
geo_idx = pd.read_csv("geoid_idx_names.csv")
geo_idx["GEOID"] = geo_idx["GEOID"].astype(int)

# ── Load ACS with raw counts ──
acs_df = pd.read_csv("acs_socioeconomic_v2.csv")
acs_df["GEOID"] = acs_df["GEOID"].astype(int)

# Merge county_idx and raw counts into cluster_df
cluster_df = cluster_df.merge(geo_idx[["GEOID", "county_idx"]], on="GEOID", how="left")
cluster_df = cluster_df.merge(
    acs_df[["GEOID", "white_pop", "pop_25plus", "bachelors_plus_count",
            "pct_no_vehicle", "insurance_coverage_pct"]],
    on="GEOID", how="left"
)

# ── Compute cluster-level rates from raw counts ──
# (sum counts across cluster, then divide — avoids Simpson's paradox)
# nchs_code already loaded from county_cluster_assignments.csv
cluster_features = cluster_df.groupby("cluster").agg(
    total_pop=("total_population", "sum"),
    total_white=("white_pop", "sum"),
    total_pop_25plus=("pop_25plus", "sum"),
    total_bachelors_plus=("bachelors_plus_count", "sum"),
    median_income=("median_household_income", "median"),
    mean_dist_to_track=("dist_to_track_mi", "mean"),
    median_pct_no_vehicle=("pct_no_vehicle", "median"),
    median_insurance=("insurance_coverage_pct", "median"),
    nchs_code=("nchs_code", "first"),  # all same within cluster
    n_counties=("GEOID", "count"),
).reset_index()

# Cluster-level percentages from aggregated counts
cluster_features["cluster_pct_white"] = (
    cluster_features["total_white"] / cluster_features["total_pop"] * 100
)
cluster_features["cluster_pct_bachelors"] = (
    cluster_features["total_bachelors_plus"] / cluster_features["total_pop_25plus"] * 100
)

# Cluster info
n_clusters = cluster_df["cluster"].nunique()
singletons = cluster_df.groupby("cluster").filter(lambda x: len(x) == 1)["cluster"].tolist()

print(f"Total clusters: {n_clusters}")
print(f"Singleton clusters: {singletons}")
print(f"\nCluster-level features:")
display(cluster_features[["cluster", "n_counties", "total_pop", "median_income",
                           "mean_dist_to_track", "cluster_pct_white",
                           "cluster_pct_bachelors"]].sort_values("median_income"))

# Sanity check
print(f"\nCluster pct_white range: {cluster_features['cluster_pct_white'].min():.1f}%–{cluster_features['cluster_pct_white'].max():.1f}%")
print(f"Cluster pct_bachelors range: {cluster_features['cluster_pct_bachelors'].min():.1f}%–{cluster_features['cluster_pct_bachelors'].max():.1f}%")

# Build cluster_indices for mobility loading
cluster_indices = {}
for c in sorted(cluster_df["cluster"].unique()):
    idx_list = cluster_df[cluster_df["cluster"] == c]["county_idx"].dropna().astype(int).values
    cluster_indices[c] = idx_list

assert cluster_df["county_idx"].notna().all(), "Some counties missing from geo_idx!" 

Total clusters: 38
Singleton clusters: [11, 17, 18, 20, 21, 22, 29, 33, 36, 37]

Cluster-level features:


,cluster,n_counties,total_pop,median_income,mean_dist_to_track,cluster_pct_white,cluster_pct_bachelors
34,34,2,54379,42012.5,60.464229,97.256294,13.979682
23,23,16,240690,45784.5,21.669718,94.616727,14.669539
37,37,1,21634,45966.0,48.939786,49.084774,11.653328
25,25,29,386837,46091.0,25.269227,64.578621,14.567594
28,28,3,44916,46706.0,3.202622,76.629709,14.928406
32,32,6,121624,46953.5,27.734990,75.882227,12.398188
16,16,8,289794,47230.5,45.252458,95.577203,16.468099
14,14,2,58632,47286.0,20.130512,57.287829,15.290638
10,10,12,350797,47309.0,32.409158,58.798678,15.864516
7,7,4,92078,47536.5,16.571338,91.278047,17.236527



Cluster pct_white range: 49.1%–97.3%
Cluster pct_bachelors range: 10.8%–38.4%


## 2. Load and Aggregate Mobility by Cluster

In [76]:
# ── Helper functions ──
def mondays_str(year, start_month=7, end_month=10):
    start = dt.date(year, start_month, 28)
    end = dt.date(year, end_month, 31)
    days_ahead = (0 - start.weekday()) % 7
    cur = start + dt.timedelta(days=days_ahead)
    out = []
    while cur <= end:
        out.append(cur.strftime("%Y%m%d"))
        cur += dt.timedelta(days=7)
    return out

mondays_2023 = mondays_str(2023)
mondays_2024 = mondays_str(2024)
all_mondays = mondays_2023 + mondays_2024

# Build date index
dates_2023 = pd.date_range(start="2023-07-31", periods=len(mondays_2023) * 7, freq="D")
dates_2024 = pd.date_range(start="2024-07-29", periods=len(mondays_2024) * 7, freq="D")
dates_all = dates_2023.union(dates_2024)

# ── For each cluster, get its county indices ──
cluster_indices = {}
for c in sorted(cluster_df["cluster"].unique()):
    idx_list = cluster_df[cluster_df["cluster"] == c]["county_idx"].dropna().astype(int).values
    cluster_indices[c] = idx_list

print(f"Date range: {dates_all[0].date()} to {dates_all[-1].date()}")
print(f"Total days: {len(dates_all)}")

Date range: 2023-07-31 to 2024-11-03
Total days: 196


In [ ]:
# ── Load precomputed flows from results/local_level/helene/ ──
# Flows were computed correctly in recompute_flows.ipynb using:
#   Within_j = sum_{d in A} M[t,:,d,j] — from cluster to A (including self)
#   Outflow_j = sum_{d not in A} M[t,:,d,j] — from cluster to outside A
#   Inflow_j = sum_{o not in A} M[t,:,j,o] — from outside A to cluster

local_results_dir = '../results/local_level/helene'

print('Loading precomputed cluster results from:', local_results_dir)
print('NOTE: If these files do not exist, run recompute_flows.ipynb first.')
print()

import os
for f in sorted(os.listdir(local_results_dir)):
    if f.endswith('.csv'):
        print(f'  {f}')


## 3. SARIMAX Baseline + Recovery Analysis per Cluster

For each cluster:
1. Fit SARIMAX on pre-Helene training data (2023 Jul-Oct + 2024 Jul–Sep 19)
2. Forecast Sep 1 – Oct 31 2024
3. Compute largest drop and recovery time
4. Plot relative difference with recovery annotation

In [ ]:
# ── SARIMAX settings ──
HRC_NAME = "helene"
LANDING_DATE = pd.Timestamp("2024-09-26")
TRAIN_END = "2024-09-19"  # before Helene impact
FORECAST_START = "2024-09-01"
FORECAST_END = "2024-10-31"
ARIMA_ORDER = (2, 0, 2)
SEASONAL_ORDER = (1, 0, 1, 7)
K_RECOVERY = 7  # consecutive days for recovery
DELTA = 0.05  # 5% threshold

FLOW_TYPES = {
    "within": cluster_within_ts,
    "inflow": cluster_inflow_ts,
    "outflow": cluster_outflow_ts,
}

all_results = {}

for flow_name, flow_data in FLOW_TYPES.items():
    print(f"\n{'#'*70}")
    print(f"### FLOW TYPE: {flow_name.upper()}")
    print(f"{'#'*70}")
    
    results_list = []
    failed_clusters = []
    
    for c in sorted(cluster_indices.keys()):
        n_counties = len(cluster_indices[c])
        is_singleton = c in singletons
        tag = " [SINGLETON]" if is_singleton else ""
        print(f"\n{'='*60}")
        print(f"  Cluster {c} ({n_counties} counties){tag} — {flow_name}")
        print(f"{'='*60}")
        
        flow_y = flow_data[c]
        cinfo = cluster_df[cluster_df["cluster"] == c]
        
        try:
            # Fit SARIMAX
            y_log, y, X = prepare_time_series_with_exog(flow_y, dates_all)
            res, y_train, X_train = fit_arimax_model(
                y_log, X,
                order=ARIMA_ORDER,
                seasonal_order=SEASONAL_ORDER,
                train_2024_end=TRAIN_END
            )
            df_rec, forecast_idx = get_predictions_and_ci(
                res, X, y,
                forecast_start=FORECAST_START,
                forecast_end=FORECAST_END
            )
            
            # ── Plot 1: Predicted vs Actual ──
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
            
            # Top: actual vs predicted with CI
            ax1.plot(df_rec.index, df_rec["y_true"], "k-", lw=1.5, label="Actual")
            ax1.plot(df_rec.index, df_rec["y_pred"], "b--", lw=1.5, label="Predicted (SARIMAX)")
            ax1.fill_between(df_rec.index, df_rec["ci_lower"], df_rec["ci_upper"],
                            color="blue", alpha=0.1, label="95% CI")
            ax1.axvline(LANDING_DATE, color="red", ls="--", lw=2, label="Helene Landing")
            ax1.set_ylabel("Flow Count")
            ax1.set_title(f"Cluster {c} — {flow_name.capitalize()} ({n_counties} counties){tag}\n"
                         f"Predicted vs Actual", fontweight="bold")
            ax1.legend(fontsize=8)
            ax1.grid(True, alpha=0.3)
            
            # Recovery time
            result = recovery_time_from_largest_drop(
                df_rec, k=K_RECOVERY, delta=DELTA,
                side="lower", landing_date=LANDING_DATE
            )
            
            # Bottom: relative difference
            rel_diff = result.get("relative_diff", None)
            rel_lower = result.get("relative_lower", None)
            rel_upper = result.get("relative_upper", None)
            
            if rel_diff is not None:
                ax2.plot(forecast_idx, rel_diff.values, "k-", lw=1.5, label="Relative diff (True-Pred)/Pred")
                ax2.axhline(0, color="gray", ls="--", lw=1)
                ax2.axhline(5, color="red", ls="--", lw=1.5, alpha=0.7)
                ax2.axhline(-5, color="red", ls="--", lw=1.5, alpha=0.7)
                if rel_lower is not None and rel_upper is not None:
                    ax2.fill_between(forecast_idx, rel_lower, rel_upper,
                                    color="red", alpha=0.1, label="95% CI (relative)")
                ax2.axvline(LANDING_DATE, color="red", ls="--", lw=2)
                
                if result["trough_date"] is not None:
                    ax2.axvline(result["trough_date"], color="orange", ls="--", lw=2, label="Trough")
                if result["recovery_date"] is not None:
                    ax2.axvline(result["recovery_date"], color="green", ls="--", lw=2,
                               label=f"Recovery ({result['recovery_days']} days)")
            
            ax2.set_ylabel("Relative Difference (%)")
            ax2.set_xlabel("Date")
            ax2.set_title("Relative Difference", fontweight="bold")
            ax2.legend(fontsize=8)
            ax2.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, "figures",
                        f"{flow_name}_cluster_{c}_pred_vs_actual.png"),
                        dpi=150, bbox_inches="tight")
            plt.close()
            
            print(f"  Trough: {result['trough_date']}")
            print(f"  Recovery: {result['recovery_date']} ({result['recovery_days']} days)")
            
            # Compute largest drop
            largest_drop_val = rel_diff.min() if rel_diff is not None else None
            
            # Get cluster-level features
            cf = cluster_features[cluster_features["cluster"] == c].iloc[0]
            results_list.append({
                    "cluster": c,
                    "n_counties": n_counties,
                    "is_singleton": is_singleton,
                    "median_income": cf["median_income"],
                    "total_pop": cf["total_pop"],
                    "mean_dist_to_track": cf["mean_dist_to_track"],
                    "cluster_pct_white": cf["cluster_pct_white"],
                    "cluster_pct_bachelors": cf["cluster_pct_bachelors"],
                    "median_pct_no_vehicle": cf["median_pct_no_vehicle"],
                    "median_insurance": cf["median_insurance"],
                    "nchs_code": cf["nchs_code"],
                    "largest_drop": largest_drop_val,
                    "trough_date": result["trough_date"],
                    "recovery_days": result["recovery_days"],
                    "recovery_date": result["recovery_date"],
                })
            
        except Exception as e:
            print(f"  FAILED: {e}")
            import traceback
            traceback.print_exc()
            failed_clusters.append(c)
            cf = cluster_features[cluster_features["cluster"] == c].iloc[0]
            results_list.append({
                    "cluster": c,
                    "n_counties": n_counties,
                    "is_singleton": is_singleton,
                    "median_income": cf["median_income"],
                    "total_pop": cf["total_pop"],
                    "mean_dist_to_track": cf["mean_dist_to_track"],
                    "cluster_pct_white": cf["cluster_pct_white"],
                    "cluster_pct_bachelors": cf["cluster_pct_bachelors"],
                    "median_pct_no_vehicle": cf["median_pct_no_vehicle"],
                    "median_insurance": cf["median_insurance"],
                    "nchs_code": cf["nchs_code"],
                    "largest_drop": None,
                    "trough_date": None,
                    "recovery_days": None,
                    "recovery_date": None,
                })
    
    rdf = pd.DataFrame(results_list).sort_values("cluster").reset_index(drop=True)
    rdf.to_csv(os.path.join(OUTPUT_DIR, f"cluster_results_{flow_name}.csv"), index=False)
    all_results[flow_name] = rdf
    
    print(f"\n{flow_name}: {len(results_list)} clusters, {len(failed_clusters)} failed")
    if failed_clusters:
        print(f"  Failed: {failed_clusters}")

In [ ]:
# ── Results summary for both flow types ──
for flow_name, rdf in all_results.items():
    print(f"\n{'='*70}")
    print(f"  {flow_name.upper()} — Cluster Results")
    print(f"{'='*70}")
    display(rdf[["cluster", "n_counties", "is_singleton", "median_income",
                  "total_pop", "mean_dist_to_track", "largest_drop", "recovery_days"]])

# Primary DVs
results_df = all_results["within"].copy()
results_inflow = all_results["inflow"].copy()

print(f"\nWithin: {results_df['largest_drop'].notna().sum()} valid drop, {results_df['recovery_days'].notna().sum()} valid recovery")
print(f"Inflow: {results_inflow['largest_drop'].notna().sum()} valid drop, {results_inflow['recovery_days'].notna().sum()} valid recovery")

## 4. Regression Analysis (All 20 Clusters)

OLS on all 20 clusters. Singletons are included but flagged.
Note: singleton estimates may be noisier.

In [ ]:
# ── OLS on ALL clusters: within AND inflow ──
# nchs_code is included as a numeric feature (1=large metro → 6=rural)
FEATURES_BASIC = ["median_income", "mean_dist_to_track", "nchs_code"]
FEATURES_FULL = ["median_income", "mean_dist_to_track", "nchs_code",
                  "cluster_pct_white", "cluster_pct_bachelors",
                  "median_pct_no_vehicle", "median_insurance"]

all_models = []

for flow_name, rdf in [("within", results_df), ("inflow", results_inflow)]:
    print(f"\n\n{'#'*70}")
    print(f"  {flow_name.upper()} FLOW — OLS REGRESSION")
    print(f"{'#'*70}")
    
    for dv_col, dv_label in [("largest_drop", "Largest Drop"), ("recovery_days", "Recovery Time")]:
        df_r = rdf.dropna(subset=[dv_col]).copy().reset_index(drop=True)
        if len(df_r) < 5:
            print(f"\n  Skipping {dv_label} — only {len(df_r)} valid observations")
            continue
        
        # Basic model: income + distance + NCHS
        print(f"\n{'='*60}")
        print(f"  {flow_name} {dv_label} ~ income + distance + nchs_code (N={len(df_r)})")
        print(f"{'='*60}")
        scaler_b = StandardScaler()
        X_b_z = pd.DataFrame(scaler_b.fit_transform(df_r[FEATURES_BASIC]), columns=FEATURES_BASIC)
        X_b = sm.add_constant(X_b_z)
        m_b = sm.OLS(df_r[dv_col], X_b).fit()
        print(m_b.summary())
        all_models.append((f"{flow_name} {dv_label} ~ basic+nchs", m_b, len(df_r)))
        
        # Full model: all features including NCHS
        print(f"\n{'='*60}")
        print(f"  {flow_name} {dv_label} ~ all features + nchs_code (N={len(df_r)})")
        print(f"{'='*60}")
        scaler_f = StandardScaler()
        X_f_z = pd.DataFrame(scaler_f.fit_transform(df_r[FEATURES_FULL]), columns=FEATURES_FULL)
        X_f = sm.add_constant(X_f_z)
        m_f = sm.OLS(df_r[dv_col], X_f).fit()
        print(m_f.summary())
        all_models.append((f"{flow_name} {dv_label} ~ full+nchs", m_f, len(df_r)))

# ── Grand comparison ──
print(f"\n\n{'='*70}")
print("ALL MODELS COMPARISON")
print("="*70)
print(f"{'Model':<55} {'N':>3} {'R²':>8} {'Adj.R²':>8} {'F p':>12}")
print("-"*86)
for name, m, n in all_models:
    print(f"{name:<55} {n:>3} {m.rsquared:>8.4f} {m.rsquared_adj:>8.4f} {m.f_pvalue:>12.4e}")

In [ ]:
# ── Bivariate correlations (ALL clusters) ──
print("="*70)
print("BIVARIATE CORRELATIONS (all clusters)")
print("="*70)

ivs = ["median_income", "mean_dist_to_track", "nchs_code",
       "cluster_pct_white", "cluster_pct_bachelors",
       "median_pct_no_vehicle", "median_insurance",
       "total_pop", "n_counties"]

for flow_name, rdf in [("within", results_df), ("inflow", results_inflow)]:
    print(f"\n--- {flow_name.upper()} ---")
    for dv in ["largest_drop", "recovery_days"]:
        valid = rdf.dropna(subset=[dv])
        if len(valid) < 4:
            continue
        print(f"\n  {dv} (N={len(valid)}):")
        for iv in ivs:
            if iv not in valid.columns:
                continue
            r_p, p_p = pearsonr(valid[iv], valid[dv])
            r_s, p_s = spearmanr(valid[iv], valid[dv])
            sig = "**" if min(p_p, p_s) < 0.05 else "*" if min(p_p, p_s) < 0.1 else ""
            print(f"    {iv:<25} r={r_p:>7.3f} (p={p_p:.3f})  ρ={r_s:>7.3f} (p={p_s:.3f}) {sig}")

## 5. Scatter Plots with LOWESS

Visualize income vs drop/recovery. Color by distance, size by population.
Include ALL clusters (singletons marked differently).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

for ax, dv, title in [
    (ax1, "largest_drop", "Largest Drop (%)"),
    (ax2, "recovery_days", "Recovery Time (days)"),
]:
    valid_all = results_df.dropna(subset=[dv])
    
    # Color by distance
    norm = mcolors.Normalize(vmin=valid_all["mean_dist_to_track"].min(),
                              vmax=valid_all["mean_dist_to_track"].max())
    
    # Plot multi-county clusters (filled circles)
    multi = valid_all[~valid_all["is_singleton"]]
    sc = ax.scatter(multi["median_income"], multi[dv],
                    s=multi["total_pop"] / 5000,
                    c=multi["mean_dist_to_track"], cmap="RdYlGn_r", norm=norm,
                    edgecolors="black", linewidth=0.8, alpha=0.8, zorder=5)
    
    # Plot singletons (open diamonds)
    single = valid_all[valid_all["is_singleton"]]
    ax.scatter(single["median_income"], single[dv],
               s=80, c=single["mean_dist_to_track"], cmap="RdYlGn_r", norm=norm,
               edgecolors="black", linewidth=1.5, alpha=0.6, zorder=4,
               marker="D")
    
    # LOWESS on multi-county only
    if len(multi) >= 5:
        lw = lowess_func(multi[dv].values, multi["median_income"].values, frac=0.6)
        ax.plot(lw[:, 0], lw[:, 1], color="blue", lw=2.5, label="LOWESS (multi-county)")
    
    # Annotate
    for _, row in valid_all.iterrows():
        marker = "*" if row["is_singleton"] else ""
        ax.annotate(f"C{int(row['cluster'])}{marker}",
                    (row["median_income"], row[dv]),
                    fontsize=7, ha="center", va="bottom",
                    xytext=(0, 6), textcoords="offset points")
    
    # Correlation (multi-county only)
    r_p, p_p = pearsonr(multi["median_income"], multi[dv])
    r_s, p_s = spearmanr(multi["median_income"], multi[dv])
    ax.text(0.02, 0.98,
            f"Multi-county only:\nPearson r={r_p:.3f} (p={p_p:.3f})\nSpearman ρ={r_s:.3f} (p={p_s:.3f})",
            transform=ax.transAxes, fontsize=9, va="top",
            bbox=dict(boxstyle="round,pad=0.3", fc="wheat", alpha=0.5))
    
    ax.set_xlabel("Cluster Median Income ($)", fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(f"Helene: Income vs {title}", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.colorbar(sc, ax=ax2, label="Mean Distance to Track (mi)", shrink=0.8)
fig.suptitle("Helene Cluster-Level Analysis (◆ = singleton clusters)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figures", "scatter_income_vs_dvs.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 6. Kruskal-Wallis Quartile Analysis

Split clusters into income quartiles and test whether drop/recovery
differs across quartiles. Uses ALL clusters (including singletons).

In [ ]:
# ── Kruskal-Wallis: income, education, race × drop/recovery × within/inflow ──

GROUPING_VARS = {
    "income_quartile": {
        "col": "median_income",
        "label": "Income",
    },
    "education_quartile": {
        "col": "cluster_pct_bachelors",
        "label": "Education (% Bachelor's+)",
    },
    "race_quartile": {
        "col": "cluster_pct_white",
        "label": "Race (% White)",
    },
    "nchs_quartile": {
        "col": "nchs_code",
        "label": "Urban-Rural (NCHS code)",
    },
}

DVs = {
    "largest_drop": "Largest Drop (%)",
    "recovery_days": "Recovery Time (days)",
}

quartile_labels = ["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]
all_kw_results = []
kw_data = {}  # store df_kw per flow for plotting

for flow_name, rdf in [("within", results_df), ("inflow", results_inflow)]:
    print(f"\n{'#'*70}")
    print(f"  {flow_name.upper()} FLOW")
    print(f"{'#'*70}")
    
    df_kw = rdf.dropna(subset=["largest_drop"]).copy().reset_index(drop=True)
    
    for grp_key, grp_cfg in GROUPING_VARS.items():
        col = grp_cfg["col"]
        label = grp_cfg["label"]
        
        df_kw[grp_key] = pd.qcut(
            df_kw[col].rank(method="first"), q=4, labels=quartile_labels
        )
        
        print(f"\n  --- {label} ---")
        for q in quartile_labels:
            sub = df_kw[df_kw[grp_key] == q]
            print(f"    {q}: n={len(sub)}, {col}={sub[col].min():.1f}–{sub[col].max():.1f}")
        
        for dv_col, dv_label in DVs.items():
            valid = df_kw.dropna(subset=[dv_col])
            groups = [valid[valid[grp_key] == q][dv_col].values for q in quartile_labels]
            groups_nonempty = [g for g in groups if len(g) > 0]
            
            kw_H, kw_p, mw_U, mw_p = np.nan, np.nan, np.nan, np.nan
            
            if len(groups_nonempty) >= 2 and all(len(g) >= 1 for g in groups_nonempty):
                kw_H, kw_p = kruskal(*groups_nonempty)
            
            q1 = valid[valid[grp_key] == "Q1 (lowest)"][dv_col].values
            q4 = valid[valid[grp_key] == "Q4 (highest)"][dv_col].values
            if len(q1) >= 2 and len(q4) >= 2:
                mw_U, mw_p = mannwhitneyu(q1, q4, alternative="two-sided")
            
            r_s, p_s = spearmanr(valid[col], valid[dv_col])
            
            sig_kw = "**" if kw_p < 0.05 else "*" if kw_p < 0.1 else ""
            sig_mw = "**" if mw_p < 0.05 else "*" if mw_p < 0.1 else ""
            print(f"\n    {flow_name} {dv_label} by {label}:")
            print(f"      KW H={kw_H:.3f}, p={kw_p:.4f} {sig_kw}")
            print(f"      MW Q1vQ4 U={mw_U:.1f}, p={mw_p:.4f} {sig_mw}")
            print(f"      Spearman ρ={r_s:.3f}, p={p_s:.4f}")
            
            all_kw_results.append({
                "flow": flow_name,
                "grouping": label,
                "dv": dv_label,
                "KW_H": round(kw_H, 4) if not np.isnan(kw_H) else None,
                "KW_p": round(kw_p, 4) if not np.isnan(kw_p) else None,
                "MW_Q1vQ4_p": round(mw_p, 4) if not np.isnan(mw_p) else None,
                "Spearman_rho": round(r_s, 3),
                "Spearman_p": round(p_s, 4),
            })
    
    kw_data[flow_name] = df_kw

kw_summary = pd.DataFrame(all_kw_results)
print(f"\n\n{'='*70}")
print("KRUSKAL-WALLIS SUMMARY (all flows × groupings × DVs)")
print("="*70)
display(kw_summary)
kw_summary.to_csv(os.path.join(OUTPUT_DIR, "kruskal_wallis_summary.csv"), index=False)

In [ ]:
# ── Boxplots: 2 flows × 3 groupings × 2 DVs ──
# Layout: one figure per flow type (2 rows × 3 cols each)

for flow_name, df_kw in kw_data.items():
    fig, axes = plt.subplots(2, 4, figsize=(26, 10))
    
    for col_idx, (grp_key, grp_cfg) in enumerate(GROUPING_VARS.items()):
        for row_idx, (dv_col, dv_label) in enumerate(DVs.items()):
            ax = axes[row_idx, col_idx]
            valid = df_kw.dropna(subset=[dv_col])
            
            box_data = [valid[valid[grp_key] == q][dv_col].dropna().values
                         for q in quartile_labels]
            
            bp = ax.boxplot(box_data, labels=["Q1", "Q2", "Q3", "Q4"],
                            patch_artist=True,
                            boxprops=dict(facecolor="lightblue", alpha=0.6),
                            medianprops=dict(color="red", linewidth=2))
            
            np.random.seed(42)
            for i, q in enumerate(quartile_labels):
                sub = valid[valid[grp_key] == q]
                jitter = np.random.uniform(-0.15, 0.15, size=len(sub))
                colors = ["red" if s else "steelblue" for s in sub["is_singleton"]]
                ax.scatter(np.full(len(sub), i + 1) + jitter, sub[dv_col],
                           s=30, c=colors, edgecolors="black", linewidth=0.3, zorder=5, alpha=0.7)
            
            # KW p-value annotation
            kw_row = kw_summary[
                (kw_summary["flow"] == flow_name) &
                (kw_summary["grouping"] == grp_cfg["label"]) &
                (kw_summary["dv"] == dv_label)
            ]
            if len(kw_row) > 0 and kw_row.iloc[0]["KW_p"] is not None:
                p_val = kw_row.iloc[0]["KW_p"]
                sig = "**" if p_val < 0.05 else "*" if p_val < 0.1 else ""
                ax.text(0.98, 0.98, f"KW p={p_val:.3f} {sig}",
                        transform=ax.transAxes, fontsize=9, va="top", ha="right",
                        bbox=dict(boxstyle="round,pad=0.3", fc="wheat", alpha=0.5))
            
            ax.set_ylabel(dv_label, fontsize=10)
            if row_idx == 0:
                ax.set_title(grp_cfg["label"], fontsize=11, fontweight="bold")
            if row_idx == 1:
                ax.set_xlabel("Quartile", fontsize=10)
            ax.grid(axis="y", alpha=0.3)
    
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="steelblue",
               markersize=8, markeredgecolor="black", label="Multi-county"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="red",
               markersize=8, markeredgecolor="black", label="Singleton"),
    ]
    axes[0, 2].legend(handles=legend_elements, fontsize=8, loc="upper right")
    
    fig.suptitle(f"Helene {flow_name.upper()}: Disruption by Income, Education, Race Quartiles",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "figures", f"boxplots_quartiles_{flow_name}.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

## 7. Summary

Key results table and comparison with Milton findings.

In [ ]:
# ── Summary table ──
print("="*70)
print("HELENE CLUSTER-LEVEL RESULTS SUMMARY")
print("="*70)

print(f"\nTotal clusters: {len(results_df)}")
print(f"Multi-county: {len(df_multi)}, Singletons: {len(df_single)}")
print(f"Failed SARIMAX: {len(failed_clusters)}")

valid = results_df.dropna(subset=["largest_drop"])
print(f"\nLargest drop range: {valid['largest_drop'].min():.1f}% to {valid['largest_drop'].max():.1f}%")
print(f"Recovery time range: {valid['recovery_days'].min():.1f} to {valid['recovery_days'].max():.1f} days")

print(f"\n{'Cluster':<10} {'N cty':>6} {'Singleton':>10} {'Med Income':>12} {'Drop (%)':>10} {'Recovery (d)':>13} {'Dist (mi)':>10}")
print("-"*71)
for _, row in results_df.sort_values("median_income").iterrows():
    s_tag = "yes" if row["is_singleton"] else ""
    drop_str = f"{row['largest_drop']:.1f}" if pd.notna(row["largest_drop"]) else "N/A"
    rec_str = f"{row['recovery_days']:.1f}" if pd.notna(row["recovery_days"]) else "N/A"
    print(f"C{int(row['cluster']):<9} {int(row['n_counties']):>6} {s_tag:>10} "
          f"${row['median_income']:>10,.0f} {drop_str:>10} {rec_str:>13} {row['mean_dist_to_track']:>10.1f}")